# SOL Hi — Model complexity analysis

This notebook analyses whether the internal validation protocol used for hyperparameter selection changes the complexity of the selected models.

The comparison is between:

- OOD holdout inner validation
- Random shuffle inner validation

The main question is:

**Does random shuffle select models that are more complex and less calibrated for final OOD test generalization?**

The analysis uses the saved per-fold artifacts:

- `params_fold_i.json`
- `complexity_fold_i.json`

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

In [2]:
PROJECT_ROOT = Path("../..").resolve()

RAW_RESULTS_DIR = PROJECT_ROOT / "results" / "hi" / "sol"

OUTPUT_DIR = (
    PROJECT_ROOT
    / "results"
    / "results_ood_vs_random_shuffle"
    / "hi"
    / "sol"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw results:", RAW_RESULTS_DIR)
print("Output dir:", OUTPUT_DIR)

Project root: /home/f.capria/drug-discovery-lohi
Raw results: /home/f.capria/drug-discovery-lohi/results/hi/sol
Output dir: /home/f.capria/drug-discovery-lohi/results/results_ood_vs_random_shuffle/hi/sol


## Experiment registry

The registry defines which result folders correspond to each combination of:

- model
- fingerprint
- protocol

In [3]:
EXPERIMENTS = [
    # Decision Tree
    {
        "model": "Decision Tree",
        "model_short": "DT",
        "fingerprint": "ECFP4",
        "protocol": "OOD holdout",
        "result_dir": "dt_sol_hi_inner_ood_holdout_ecfp4",
    },
    {
        "model": "Decision Tree",
        "model_short": "DT",
        "fingerprint": "ECFP4",
        "protocol": "Random shuffle",
        "result_dir": "dt_sol_hi_random_shuffle_ecfp4",
    },
    {
        "model": "Decision Tree",
        "model_short": "DT",
        "fingerprint": "MACCS",
        "protocol": "OOD holdout",
        "result_dir": "dt_sol_hi_inner_ood_holdout_maccs",
    },
    {
        "model": "Decision Tree",
        "model_short": "DT",
        "fingerprint": "MACCS",
        "protocol": "Random shuffle",
        "result_dir": "dt_sol_hi_random_shuffle_maccs",
    },

    # Logistic Regression
    {
        "model": "Logistic Regression",
        "model_short": "LR",
        "fingerprint": "ECFP4",
        "protocol": "OOD holdout",
        "result_dir": "lr_sol_hi_inner_ood_holdout_ecfp4",
    },
    {
        "model": "Logistic Regression",
        "model_short": "LR",
        "fingerprint": "ECFP4",
        "protocol": "Random shuffle",
        "result_dir": "lr_sol_hi_random_shuffle_ecfp4",
    },
    {
        "model": "Logistic Regression",
        "model_short": "LR",
        "fingerprint": "MACCS",
        "protocol": "OOD holdout",
        "result_dir": "lr_sol_hi_inner_ood_holdout_maccs",
    },
    {
        "model": "Logistic Regression",
        "model_short": "LR",
        "fingerprint": "MACCS",
        "protocol": "Random shuffle",
        "result_dir": "lr_sol_hi_random_shuffle_maccs",
    },
    {
        "model": "Logistic Regression",
        "model_short": "LR",
        "fingerprint": "RDKit desc",
        "protocol": "OOD holdout",
        "result_dir": "lr_sol_hi_inner_ood_holdout_rdkit_desc",
    },
    {
        "model": "Logistic Regression",
        "model_short": "LR",
        "fingerprint": "RDKit desc",
        "protocol": "Random shuffle",
        "result_dir": "lr_sol_hi_random_shuffle_rdkit_desc",
    },

    # Linear SVM
    {
        "model": "SVM linear",
        "model_short": "SVM",
        "fingerprint": "ECFP4",
        "protocol": "OOD holdout",
        "result_dir": "svm_linear_sol_hi_inner_ood_holdout_ecfp4",
    },
    {
        "model": "SVM linear",
        "model_short": "SVM",
        "fingerprint": "ECFP4",
        "protocol": "Random shuffle",
        "result_dir": "svm_linear_sol_hi_random_shuffle_ecfp4",
    },
    {
        "model": "SVM linear",
        "model_short": "SVM",
        "fingerprint": "MACCS",
        "protocol": "OOD holdout",
        "result_dir": "svm_linear_sol_hi_inner_ood_holdout_maccs",
    },
    {
        "model": "SVM linear",
        "model_short": "SVM",
        "fingerprint": "MACCS",
        "protocol": "Random shuffle",
        "result_dir": "svm_linear_sol_hi_random_shuffle_maccs",
    },
]

In [4]:
def read_json(path):
    with open(path, "r") as f:
        return json.load(f)


def safe_get(dct, key, default=np.nan):
    return dct.get(key, default)

In [5]:
def load_complexity_rows(experiments, raw_results_dir):
    rows = []

    for exp in experiments:
        result_dir = raw_results_dir / exp["result_dir"]

        if not result_dir.exists():
            print(f"Missing directory: {result_dir}")
            continue

        for fold in [1, 2, 3]:
            params_path = result_dir / f"params_fold_{fold}.json"
            complexity_path = result_dir / f"complexity_fold_{fold}.json"

            if not params_path.exists():
                print(f"Missing params file: {params_path}")
                continue

            if not complexity_path.exists():
                print(f"Missing complexity file: {complexity_path}")
                continue

            params = read_json(params_path)
            complexity = read_json(complexity_path)

            train_metrics = params.get("train_metrics", {})
            test_metrics = params.get("test_metrics", {})

            inner_pr_auc = params.get("inner_selection_score", np.nan)
            inner_train_pr_auc = params.get("inner_train_score", np.nan)
            train_pr_auc = train_metrics.get("pr_auc", np.nan)
            test_pr_auc = test_metrics.get("pr_auc", np.nan)

            row = {
                "model": exp["model"],
                "model_short": exp["model_short"],
                "fingerprint": exp["fingerprint"],
                "protocol": exp["protocol"],
                "result_dir": exp["result_dir"],
                "fold": fold,

                # Scores
                "inner_pr_auc": inner_pr_auc,
                "inner_train_pr_auc": inner_train_pr_auc,
                "train_pr_auc": train_pr_auc,
                "test_pr_auc": test_pr_auc,

                # Gaps
                "train_inner_gap": train_pr_auc - inner_pr_auc,
                "inner_test_gap": inner_pr_auc - test_pr_auc,
                "train_test_gap": train_pr_auc - test_pr_auc,
            }

            # Add all complexity fields
            for key, value in complexity.items():
                row[key] = value

            rows.append(row)

    return pd.DataFrame(rows)

In [6]:
complexity_all = load_complexity_rows(EXPERIMENTS, RAW_RESULTS_DIR)

complexity_all = complexity_all.sort_values(
    ["model_short", "fingerprint", "protocol", "fold"]
).reset_index(drop=True)

complexity_all.head(3)

,model,model_short,fingerprint,protocol,result_dir,fold,inner_pr_auc,inner_train_pr_auc,train_pr_auc,test_pr_auc,...,l1_norm,l2_norm,approx_margin,intercept,C,penalty,l1_ratio,dual,n_support_per_class,n_support_total
0,Decision Tree,DT,ECFP4,OOD holdout,dt_sol_hi_inner_ood_holdout_ecfp4,1,0.390689,0.589258,0.5752,0.3182,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Decision Tree,DT,ECFP4,OOD holdout,dt_sol_hi_inner_ood_holdout_ecfp4,2,0.378160,0.657379,0.7060,0.2997,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Decision Tree,DT,ECFP4,OOD holdout,dt_sol_hi_inner_ood_holdout_ecfp4,3,0.372324,0.798692,0.7555,0.2934,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
print("Rows loaded:", len(complexity_all))
print("Expected rows:", len(EXPERIMENTS) * 3)

complexity_all[
    [
        "model",
        "fingerprint",
        "protocol",
        "fold",
        "inner_pr_auc",
        "train_pr_auc",
        "test_pr_auc",
        "inner_test_gap",
        "train_test_gap",
        "model_class",
    ]
]

Rows loaded: 42
Expected rows: 42


,model,fingerprint,protocol,fold,inner_pr_auc,train_pr_auc,test_pr_auc,inner_test_gap,train_test_gap,model_class
0,Decision Tree,ECFP4,OOD holdout,1,0.390689,0.5752,0.3182,0.072489,0.2570,DecisionTreeClassifier
1,Decision Tree,ECFP4,OOD holdout,2,0.378160,0.7060,0.2997,0.078460,0.4063,DecisionTreeClassifier
2,Decision Tree,ECFP4,OOD holdout,3,0.372324,0.7555,0.2934,0.078924,0.4621,DecisionTreeClassifier
3,Decision Tree,ECFP4,Random shuffle,1,0.533999,0.8287,0.3107,0.223299,0.5180,DecisionTreeClassifier
4,Decision Tree,ECFP4,Random shuffle,2,0.489036,0.8015,0.3009,0.188136,0.5006,DecisionTreeClassifier
5,Decision Tree,ECFP4,Random shuffle,3,0.421861,0.8464,0.2806,0.141261,0.5658,DecisionTreeClassifier
6,Decision Tree,MACCS,OOD holdout,1,0.396868,0.5089,0.3378,0.059068,0.1711,DecisionTreeClassifier
7,Decision Tree,MACCS,OOD holdout,2,0.381122,0.4459,0.3459,0.035222,0.1000,DecisionTreeClassifier
8,Decision Tree,MACCS,OOD holdout,3,0.390495,0.5326,0.3142,0.076295,0.2184,DecisionTreeClassifier
9,Decision Tree,MACCS,Random shuffle,1,0.466677,0.5965,0.4016,0.065077,0.1949,DecisionTreeClassifier


## Logistic Regression complexity table

For Logistic Regression complexity is described by:

- selected `C` - larger --> weaker regularization
- selected `l1_ratio`
- number of non-zero coefficients
- sparsity
- L1 norm of the coefficient vector
- L2 norm of the coefficient vector - 

In [8]:
lr_cols = [
    "model",
    "fingerprint",
    "protocol",
    "fold",
    "C",
    "l1_ratio",
    "class_weight",
    "n_nonzero_coefficients",
    "sparsity",
    "l1_norm",
    "l2_norm",
    "inner_pr_auc",
    "test_pr_auc",
    "inner_test_gap",
    "train_test_gap",
]

lr_table = (
    complexity_all[complexity_all["model_short"] == "LR"]
    [lr_cols]
    .sort_values(["fingerprint", "protocol", "fold"])
    .reset_index(drop=True)
)

lr_table

,model,fingerprint,protocol,fold,C,l1_ratio,class_weight,n_nonzero_coefficients,sparsity,l1_norm,l2_norm,inner_pr_auc,test_pr_auc,inner_test_gap,train_test_gap
0,Logistic Regression,ECFP4,OOD holdout,1,0.050,0.00,NaN,1024.0,0.000000,68.586399,2.896540,0.475664,0.5053,-0.029636,0.3449
1,Logistic Regression,ECFP4,OOD holdout,2,0.100,0.50,balanced,131.0,0.872070,24.194311,2.732262,0.490251,0.5072,-0.016949,0.1897
2,Logistic Regression,ECFP4,OOD holdout,3,0.500,0.50,NaN,426.0,0.583984,120.670767,7.512598,0.541434,0.4208,0.120634,0.5232
3,Logistic Regression,ECFP4,Random shuffle,1,0.100,0.00,balanced,1024.0,0.000000,124.166675,5.009277,0.565461,0.4876,0.077861,0.4168
4,Logistic Regression,ECFP4,Random shuffle,2,0.050,0.00,NaN,1024.0,0.000000,69.250098,2.906760,0.637104,0.5271,0.110004,0.3257
5,Logistic Regression,ECFP4,Random shuffle,3,0.500,1.00,NaN,224.0,0.781250,74.654154,6.536329,0.548701,0.4314,0.117301,0.4468
6,Logistic Regression,MACCS,OOD holdout,1,0.100,0.25,NaN,77.0,0.538922,11.844633,1.890987,0.498532,0.4671,0.031432,0.1273
7,Logistic Regression,MACCS,OOD holdout,2,10.000,0.50,NaN,141.0,0.155689,104.996958,12.010972,0.491338,0.5009,-0.009562,0.2016
8,Logistic Regression,MACCS,OOD holdout,3,0.500,0.00,NaN,146.0,0.125749,45.910097,5.005261,0.470671,0.4623,0.008371,0.2262
9,Logistic Regression,MACCS,Random shuffle,1,0.005,0.00,balanced,147.0,0.119760,5.802065,0.692018,0.489417,0.4312,0.058217,0.0888


## Linear SVM complexity table

For linear SVM the indicators are:

- selected `C`
- L2 norm of the weight vector
- approximate margin

In [9]:
svm_cols = [
    "model",
    "fingerprint",
    "protocol",
    "fold",
    "C",
    "class_weight",
    "l2_norm",
    "approx_margin",
    "n_nonzero_coefficients",
    "inner_pr_auc",
    "test_pr_auc",
    "inner_test_gap",
    "train_test_gap",
]

svm_table = (
    complexity_all[complexity_all["model_short"] == "SVM"]
    [svm_cols]
    .sort_values(["fingerprint", "protocol", "fold"])
    .reset_index(drop=True)
)

svm_table

,model,fingerprint,protocol,fold,C,class_weight,l2_norm,approx_margin,n_nonzero_coefficients,inner_pr_auc,test_pr_auc,inner_test_gap,train_test_gap
0,SVM linear,ECFP4,OOD holdout,1,0.010,balanced,2.054727,0.486683,1024.0,0.464509,0.5082,-0.043691,0.2691
1,SVM linear,ECFP4,OOD holdout,2,0.005,balanced,1.411017,0.708709,1024.0,0.452877,0.5282,-0.075323,0.1896
2,SVM linear,ECFP4,OOD holdout,3,0.010,balanced,2.025025,0.493821,1023.0,0.520898,0.4539,0.066998,0.3256
3,SVM linear,ECFP4,Random shuffle,1,0.100,NaN,5.042986,0.198295,1023.0,0.560148,0.3956,0.164548,0.5454
4,SVM linear,ECFP4,Random shuffle,2,0.010,balanced,2.051234,0.487511,1023.0,0.655910,0.5394,0.116510,0.2343
5,SVM linear,ECFP4,Random shuffle,3,0.010,balanced,2.024969,0.493835,1023.0,0.549668,0.4538,0.095868,0.3256
6,SVM linear,MACCS,OOD holdout,1,0.010,NaN,0.313501,3.189784,143.0,0.487033,0.4933,-0.006267,0.1360
7,SVM linear,MACCS,OOD holdout,2,0.500,balanced,5.479380,0.182502,147.0,0.495652,0.4753,0.020352,0.1264
8,SVM linear,MACCS,OOD holdout,3,0.050,NaN,1.798084,0.556147,144.0,0.461430,0.4581,0.003330,0.1989
9,SVM linear,MACCS,Random shuffle,1,0.005,balanced,0.958080,1.043754,147.0,0.499009,0.4409,0.058109,0.0710


## Decision Tree complexity table

For Decision Trees:

- selected `ccp_alpha`
- selected `max_depth`
- effective tree depth
- number of nodes
- number of leaves
- number of features used in the tree
- average minimum depth of the used features

The number of nodes and leaves gives a direct indication of how large the fitted tree is.

In [10]:
dt_cols = [
    "model",
    "fingerprint",
    "protocol",
    "fold",
    "ccp_alpha",
    "max_depth",
    "effective_depth",
    "n_nodes",
    "n_leaves",
    "n_features_used",
    "used_feature_fraction",
    "feature_min_depth_mean",
    "feature_min_depth_std",
    "inner_pr_auc",
    "test_pr_auc",
    "inner_test_gap",
    "train_test_gap",
]

dt_table = (
    complexity_all[complexity_all["model_short"] == "DT"]
    [dt_cols]
    .sort_values(["fingerprint", "protocol", "fold"])
    .reset_index(drop=True)
)

dt_table

,model,fingerprint,protocol,fold,ccp_alpha,max_depth,effective_depth,n_nodes,n_leaves,n_features_used,used_feature_fraction,feature_min_depth_mean,feature_min_depth_std,inner_pr_auc,test_pr_auc,inner_test_gap,train_test_gap
0,Decision Tree,ECFP4,OOD holdout,1,0.0010,20.0,20.0,199.0,100.0,90.0,0.087891,10.033333,4.517743,0.390689,0.3182,0.072489,0.2570
1,Decision Tree,ECFP4,OOD holdout,2,0.0000,15.0,15.0,165.0,83.0,60.0,0.058594,5.950000,3.201172,0.378160,0.2997,0.078460,0.4063
2,Decision Tree,ECFP4,OOD holdout,3,0.0010,20.0,15.0,127.0,64.0,56.0,0.054688,6.285714,3.320468,0.372324,0.2934,0.078924,0.4621
3,Decision Tree,ECFP4,Random shuffle,1,0.0000,15.0,15.0,223.0,112.0,90.0,0.087891,8.033333,3.745961,0.533999,0.3107,0.223299,0.5180
4,Decision Tree,ECFP4,Random shuffle,2,0.0000,15.0,15.0,359.0,180.0,149.0,0.145508,8.315436,3.337980,0.489036,0.3009,0.188136,0.5006
5,Decision Tree,ECFP4,Random shuffle,3,0.0000,15.0,15.0,159.0,80.0,74.0,0.072266,7.162162,3.357231,0.421861,0.2806,0.141261,0.5658
6,Decision Tree,MACCS,OOD holdout,1,0.0010,5.0,5.0,37.0,19.0,17.0,0.101796,2.764706,1.214104,0.396868,0.3378,0.059068,0.1711
7,Decision Tree,MACCS,OOD holdout,2,0.0000,5.0,5.0,57.0,29.0,26.0,0.155689,3.000000,1.109400,0.381122,0.3459,0.035222,0.1000
8,Decision Tree,MACCS,OOD holdout,3,0.0000,10.0,10.0,201.0,101.0,67.0,0.401198,5.552239,2.280898,0.390495,0.3142,0.076295,0.2184
9,Decision Tree,MACCS,Random shuffle,1,0.0000,10.0,10.0,201.0,101.0,65.0,0.389222,5.369231,2.208858,0.466677,0.4016,0.065077,0.1949


## Gap analysis table

This table collects the three main performance levels:

- train PR-AUC
- inner validation PR-AUC
- final OOD test PR-AUC

and the corresponding gaps:

$$\text{train-inner gap} = \text{train PR-AUC} - \text{inner PR-AUC}$$

$$\text{inner-test gap} = \text{inner PR-AUC} - \text{test PR-AUC}$$

$$\text{train-test gap} = \text{train PR-AUC} - \text{test PR-AUC}$$


This table connects model selection, overfitting and OOD generalization.

In [11]:
gap_cols = [
    "model",
    "model_short",
    "fingerprint",
    "protocol",
    "fold",
    "train_pr_auc",
    "inner_pr_auc",
    "inner_train_pr_auc",
    "test_pr_auc",
    "train_inner_gap",
    "inner_test_gap",
    "train_test_gap",
]

gap_analysis = (
    complexity_all[gap_cols]
    .sort_values(["model_short", "fingerprint", "protocol", "fold"])
    .reset_index(drop=True)
)

gap_analysis

,model,model_short,fingerprint,protocol,fold,train_pr_auc,inner_pr_auc,inner_train_pr_auc,test_pr_auc,train_inner_gap,inner_test_gap,train_test_gap
0,Decision Tree,DT,ECFP4,OOD holdout,1,0.5752,0.390689,0.589258,0.3182,0.184511,0.072489,0.2570
1,Decision Tree,DT,ECFP4,OOD holdout,2,0.7060,0.378160,0.657379,0.2997,0.327840,0.078460,0.4063
2,Decision Tree,DT,ECFP4,OOD holdout,3,0.7555,0.372324,0.798692,0.2934,0.383176,0.078924,0.4621
3,Decision Tree,DT,ECFP4,Random shuffle,1,0.8287,0.533999,0.795823,0.3107,0.294701,0.223299,0.5180
4,Decision Tree,DT,ECFP4,Random shuffle,2,0.8015,0.489036,0.702036,0.3009,0.312464,0.188136,0.5006
5,Decision Tree,DT,ECFP4,Random shuffle,3,0.8464,0.421861,0.873637,0.2806,0.424539,0.141261,0.5658
6,Decision Tree,DT,MACCS,OOD holdout,1,0.5089,0.396868,0.629069,0.3378,0.112032,0.059068,0.1711
7,Decision Tree,DT,MACCS,OOD holdout,2,0.4459,0.381122,0.493860,0.3459,0.064778,0.035222,0.1000
8,Decision Tree,DT,MACCS,OOD holdout,3,0.5326,0.390495,0.607051,0.3142,0.142105,0.076295,0.2184
9,Decision Tree,DT,MACCS,Random shuffle,1,0.5965,0.466677,0.561701,0.4016,0.129823,0.065077,0.1949


## Aggregated complexity summary


In [12]:
summary_metrics = [
    "inner_pr_auc",
    "test_pr_auc",
    "inner_test_gap",
    "train_test_gap",
    "l2_norm",
    "n_nonzero_coefficients",
    "approx_margin",
    "effective_depth",
    "n_nodes",
    "n_leaves",
    "n_features_used",
    "feature_min_depth_mean",
]

available_summary_metrics = [
    col for col in summary_metrics
    if col in complexity_all.columns
]

complexity_summary = (
    complexity_all
    .groupby(["model", "model_short", "fingerprint", "protocol"], as_index=False)
    [available_summary_metrics]
    .agg(["mean", "std"])
)

complexity_summary.columns = [
    "_".join([c for c in col if c])
    for col in complexity_summary.columns
]

complexity_summary = complexity_summary.reset_index()

complexity_summary

,index,model,model_short,fingerprint,protocol,inner_pr_auc_mean,inner_pr_auc_std,test_pr_auc_mean,test_pr_auc_std,inner_test_gap_mean,...,effective_depth_mean,effective_depth_std,n_nodes_mean,n_nodes_std,n_leaves_mean,n_leaves_std,n_features_used_mean,n_features_used_std,feature_min_depth_mean_mean,feature_min_depth_mean_std
0,0,Decision Tree,DT,ECFP4,OOD holdout,0.380391,0.009384,0.303767,0.012890,0.076624,...,16.666667,2.886751,163.666667,36.018514,82.333333,18.009257,68.666667,18.583146,7.423016,2.266825
1,1,Decision Tree,DT,ECFP4,Random shuffle,0.481632,0.056435,0.297400,0.015352,0.184232,...,15.000000,0.000000,247.000000,102.137163,124.000000,51.068581,104.333333,39.501055,7.836977,0.601188
2,2,Decision Tree,DT,MACCS,OOD holdout,0.389495,0.007921,0.332633,0.016469,0.056862,...,6.666667,2.886751,98.333333,89.472528,49.666667,44.736264,36.666667,26.652079,3.772315,1.545942
3,3,Decision Tree,DT,MACCS,Random shuffle,0.476626,0.019052,0.355300,0.047016,0.121326,...,12.666667,2.309401,197.000000,48.124838,99.000000,24.062419,65.333333,9.504385,5.855855,0.634379
4,4,Logistic Regression,LR,ECFP4,OOD holdout,0.502449,0.034540,0.477767,0.049344,0.024683,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,5,Logistic Regression,LR,ECFP4,Random shuffle,0.583755,0.046955,0.482033,0.048092,0.101722,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,6,Logistic Regression,LR,MACCS,OOD holdout,0.486847,0.014463,0.476767,0.021037,0.010081,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,7,Logistic Regression,LR,MACCS,Random shuffle,0.541113,0.046275,0.461167,0.033227,0.079946,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,8,Logistic Regression,LR,RDKit desc,OOD holdout,0.582203,0.019845,0.590767,0.040560,-0.008564,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,9,Logistic Regression,LR,RDKit desc,Random shuffle,0.643298,0.038035,0.587533,0.027067,0.055764,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
complexity_all.to_csv(OUTPUT_DIR / "complexity_all.csv", index=False)
lr_table.to_csv(OUTPUT_DIR / "complexity_lr.csv", index=False)
svm_table.to_csv(OUTPUT_DIR / "complexity_svm.csv", index=False)
dt_table.to_csv(OUTPUT_DIR / "complexity_dt.csv", index=False)
gap_analysis.to_csv(OUTPUT_DIR / "complexity_gap_analysis.csv", index=False)
complexity_summary.to_csv(OUTPUT_DIR / "complexity_summary.csv", index=False)

print("Saved complexity tables in:")
print(OUTPUT_DIR)

Saved complexity tables in:
/home/f.capria/drug-discovery-lohi/results/results_ood_vs_random_shuffle/hi/sol
